# Non-Training Validation Review

这个 notebook 改成了可重跑版。

主流程不再直接读取现成 CSV，而是：
- 直接读取 `gsm8k/main` 和 `gsm8k/socratic` 的原始 parquet
- 直接加载本地模型 checkpoint
- 重新计算 analysis-only 指标
- 在 notebook 内部生成总表和结论

已有 CSV 只用于可选导出，不作为主输入。


## 0. 先用通俗的话说这个方法

我们的方法，可以先不用英文名，直接理解成一句话：

> 我们想让模型对“同一道题的不同写法”更稳定，但不要把“答案刚好一样”这种假信号也学进去。

这里最关键的是三件事：

- `selected-no-neg`：会筛一批坐标，但不做负控过滤
- `selected-neg`：筛坐标时额外排除“不同题但同答案也稳定”的坐标
- `random`：随机选一批坐标，当最弱对照

这里的“负控”不是错误标签，也不是噪声，而是：
- 两道不同题
- 但最终答案一样

如果某个内部坐标在这种情况下也很稳定，那它更像 answer cue，而不是 reasoning signal。

所以这个 notebook 主要回答的是一个前置问题：

> 负控这个筛选规则本身，能不能稳定找到更干净的内部坐标？


In [4]:
from pathlib import Path
from itertools import combinations
import sys
import json
import random

import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd()
SCRIPTS = ROOT / 'scripts'
if str(SCRIPTS) not in sys.path:
    sys.path.append(str(SCRIPTS))

from pilot_negative_control_analysis import (
    load_gsm8k_orbit_pairs,
    choose_device,
    load_local_model,
    collect_features,
    zscore_basis,
    compute_coordinate_statistics,
    select_coordinate_sets,
    summarize_coordinate_set,
    compute_answer_logit_drop_for_set,
    evaluate_generation_consistency,
)

ROOT

PosixPath('/storage/zyj_data/latent_idea')

In [5]:
DATASET_ROOT = ROOT / 'huggingface_datasets'
MODEL_GPT2 = ROOT / 'huggingface_models' / 'gpt2'
MODEL_LLAMA = ROOT / 'huggingface_models' / 'Llama-3.2-1B-Instruct'

DEFAULT_DEVICE = choose_device(None)
print('default device:', DEFAULT_DEVICE)

# 为了让 notebook 默认可跑，先提供一个 core 配置集。
# 如果你要完整复现当前 analysis-only 表，请把 ACTIVE_CONFIGS 改成 FULL_CONFIGS。

CORE_CONFIGS = [
    dict(label='Llama1B train seed42 k32 layer-1', model_path=MODEL_LLAMA, split='train', seed=42, top_k=32, num_samples=32, layer_index=-1, do_generation_eval=True),
    dict(label='Llama1B train seed7 k32 layer-1', model_path=MODEL_LLAMA, split='train', seed=7, top_k=32, num_samples=32, layer_index=-1, do_generation_eval=False),
    dict(label='Llama1B train seed42 k16 layer-1', model_path=MODEL_LLAMA, split='train', seed=42, top_k=16, num_samples=32, layer_index=-1, do_generation_eval=False),
    dict(label='Llama1B train seed42 k32 layer-2', model_path=MODEL_LLAMA, split='train', seed=42, top_k=32, num_samples=32, layer_index=-2, do_generation_eval=False),
    dict(label='Llama1B test seed42 k32 layer-1', model_path=MODEL_LLAMA, split='test', seed=42, top_k=32, num_samples=16, layer_index=-1, do_generation_eval=False),
]

FULL_CONFIGS = CORE_CONFIGS + [
    dict(label='gpt2 train seed42 k32 layer-1', model_path=MODEL_GPT2, split='train', seed=42, top_k=32, num_samples=64, layer_index=-1, do_generation_eval=True),
    dict(label='gpt2 train seed7 k32 layer-1', model_path=MODEL_GPT2, split='train', seed=7, top_k=32, num_samples=64, layer_index=-1, do_generation_eval=False),
    dict(label='gpt2 train seed123 k32 layer-1', model_path=MODEL_GPT2, split='train', seed=123, top_k=32, num_samples=64, layer_index=-1, do_generation_eval=False),
    dict(label='gpt2 train seed42 k16 layer-1', model_path=MODEL_GPT2, split='train', seed=42, top_k=16, num_samples=64, layer_index=-1, do_generation_eval=False),
    dict(label='gpt2 train seed42 k64 layer-1', model_path=MODEL_GPT2, split='train', seed=42, top_k=64, num_samples=64, layer_index=-1, do_generation_eval=False),
    dict(label='gpt2 train seed42 k32 layer-2', model_path=MODEL_GPT2, split='train', seed=42, top_k=32, num_samples=64, layer_index=-2, do_generation_eval=False),
    dict(label='gpt2 train seed42 k32 layer-4', model_path=MODEL_GPT2, split='train', seed=42, top_k=32, num_samples=64, layer_index=-4, do_generation_eval=False),
    dict(label='gpt2 train seed42 k32 layer-8', model_path=MODEL_GPT2, split='train', seed=42, top_k=32, num_samples=64, layer_index=-8, do_generation_eval=False),
    dict(label='gpt2 test seed42 k32 layer-1', model_path=MODEL_GPT2, split='test', seed=42, top_k=32, num_samples=64, layer_index=-1, do_generation_eval=False),
]

ACTIVE_CONFIGS = CORE_CONFIGS
pd.DataFrame(ACTIVE_CONFIGS)[['label', 'split', 'seed', 'top_k', 'num_samples', 'layer_index', 'do_generation_eval']]

default device: cuda


,label,split,seed,top_k,num_samples,layer_index,do_generation_eval
0,Llama1B train seed42 k32 layer-1,train,42,32,32,-1,True
1,Llama1B train seed7 k32 layer-1,train,7,32,32,-1,False
2,Llama1B train seed42 k16 layer-1,train,42,16,32,-1,False
3,Llama1B train seed42 k32 layer-2,train,42,32,32,-2,False
4,Llama1B test seed42 k32 layer-1,test,42,32,16,-1,False


## 1. 运行说明

- 如果你只想先检查 strong model 的趋势，保留 `ACTIVE_CONFIGS = CORE_CONFIGS`。
- 如果你要完整复现当前 analysis-only 总表，把 `ACTIVE_CONFIGS` 改成 `FULL_CONFIGS`。
- 这个 notebook 会真实加载模型并重跑 hidden-state 分析，所以比直接读 CSV 慢很多。


In [6]:
MODEL_CACHE = {}

def get_model_bundle(model_path: Path, device: str):
    key = (str(model_path), device)
    if key not in MODEL_CACHE:
        print(f'loading model: {model_path} on {device}')
        MODEL_CACHE[key] = load_local_model(model_path, device=device)
    return MODEL_CACHE[key]


def run_analysis_config(config: dict, device: str = None):
    device = choose_device(device)
    tokenizer, model = get_model_bundle(config['model_path'], device)

    pairs = load_gsm8k_orbit_pairs(
        dataset_root=DATASET_ROOT,
        split=config['split'],
        max_pairs=config['num_samples'],
        seed=config['seed'],
    )

    features = collect_features(
        model=model,
        tokenizer=tokenizer,
        pairs=pairs,
        device=device,
        max_length=384,
        layer_index=config['layer_index'],
        do_generation_eval=config['do_generation_eval'],
        generation_tokens=16,
    )

    mu, sigma = zscore_basis(features)
    stats = compute_coordinate_statistics(
        model=model,
        examples=features,
        mu=mu,
        sigma=sigma,
        max_pairs_per_answer=8,
        seed=config['seed'],
    )

    selected_sets = select_coordinate_sets(
        stats=stats,
        top_k=min(config['top_k'], len(stats['orbit_stability'])),
        seed=config['seed'],
        same_answer_filter_quantile=0.75,
    )

    summaries = {}
    for name in ['selected_neg', 'selected_no_neg', 'high_activation', 'random']:
        summaries[name] = summarize_coordinate_set(
            name=name,
            indices=selected_sets[name],
            stats=stats,
            answer_logit_drop=compute_answer_logit_drop_for_set(model, features, selected_sets[name]),
        )

    generation_summary = evaluate_generation_consistency(features)
    row = {
        'label': config['label'],
        'model': 'Llama1B' if 'Llama-3.2-1B-Instruct' in str(config['model_path']) else 'gpt2',
        'split': config['split'],
        'seed': config['seed'],
        'top_k': config['top_k'],
        'num_samples': config['num_samples'],
        'layer_index': config['layer_index'],
        'selected_neg_omsa': summaries['selected_neg']['orbit_minus_same_answer'],
        'selected_no_neg_omsa': summaries['selected_no_neg']['orbit_minus_same_answer'],
        'random_omsa': summaries['random']['orbit_minus_same_answer'],
        'delta_neg_minus_no_neg': summaries['selected_neg']['orbit_minus_same_answer'] - summaries['selected_no_neg']['orbit_minus_same_answer'],
        'delta_neg_minus_random': summaries['selected_neg']['orbit_minus_same_answer'] - summaries['random']['orbit_minus_same_answer'],
        'selected_neg_same_answer': summaries['selected_neg']['same_answer_stability_mean'],
        'selected_no_neg_same_answer': summaries['selected_no_neg']['same_answer_stability_mean'],
        'selected_neg_logit_drop': summaries['selected_neg']['answer_logit_drop_mean'],
        'selected_no_neg_logit_drop': summaries['selected_no_neg']['answer_logit_drop_mean'],
        'main_accuracy': generation_summary.get('main_accuracy'),
        'socratic_accuracy': generation_summary.get('socratic_accuracy'),
        'rewrite_consistency': generation_summary.get('rewrite_consistency'),
        'rewrite_both_correct': generation_summary.get('rewrite_both_correct'),
    }
    return {
        'config': config,
        'row': row,
        'selected_sets': {k: v.tolist() if hasattr(v, 'tolist') else v for k, v in selected_sets.items()},
        'summaries': summaries,
        'generation_summary': generation_summary,
        'pairs': pairs,
        'features': features,
    }


def compute_overlap_rows(run_outputs):
    rows = []
    for a, b in combinations(run_outputs, 2):
        if a['row']['model'] != b['row']['model']:
            continue
        set_a = set(a['selected_sets']['selected_neg'])
        set_b = set(b['selected_sets']['selected_neg'])
        inter = len(set_a & set_b)
        union = len(set_a | set_b)
        rows.append({
            'run_a': a['row']['label'],
            'run_b': b['row']['label'],
            'model': a['row']['model'],
            'intersection': inter,
            'union': union,
            'jaccard': inter / union if union else 0.0,
        })
    return pd.DataFrame(rows)


In [7]:
run_outputs = []
for cfg in ACTIVE_CONFIGS:
    print('running:', cfg['label'])
    out = run_analysis_config(cfg)
    run_outputs.append(out)

table_df = pd.DataFrame([out['row'] for out in run_outputs]).sort_values(['model', 'split', 'seed', 'top_k', 'layer_index', 'label']).reset_index(drop=True)
overlap_df = compute_overlap_rows(run_outputs).sort_values(['model', 'run_a', 'run_b']).reset_index(drop=True)

print('finished runs:', len(run_outputs))
table_df.head()

running: Llama1B train seed42 k32 layer-1
loading model: /storage/zyj_data/latent_idea/huggingface_models/Llama-3.2-1B-Instruct on cuda


`torch_dtype` is deprecated! Use `dtype` instead!


running: Llama1B train seed7 k32 layer-1
running: Llama1B train seed42 k16 layer-1
running: Llama1B train seed42 k32 layer-2
running: Llama1B test seed42 k32 layer-1
finished runs: 5


,label,model,split,seed,top_k,num_samples,layer_index,selected_neg_omsa,selected_no_neg_omsa,random_omsa,delta_neg_minus_no_neg,delta_neg_minus_random,selected_neg_same_answer,selected_no_neg_same_answer,selected_neg_logit_drop,selected_no_neg_logit_drop,main_accuracy,socratic_accuracy,rewrite_consistency,rewrite_both_correct
0,Llama1B test seed42 k32 layer-1,Llama1B,test,42,32,16,-1,0.284572,0.228589,0.111626,0.055983,0.172946,0.479293,0.541445,3.117866,3.287113,NaN,NaN,NaN,NaN
1,Llama1B train seed7 k32 layer-1,Llama1B,train,7,32,32,-1,0.264731,0.246013,0.101140,0.018719,0.163591,0.491194,0.509678,3.373740,3.644687,NaN,NaN,NaN,NaN
2,Llama1B train seed42 k16 layer-1,Llama1B,train,42,16,32,-1,0.277310,0.184822,0.153060,0.092488,0.124249,0.481921,0.579034,1.598073,2.105577,NaN,NaN,NaN,NaN
3,Llama1B train seed42 k32 layer-2,Llama1B,train,42,32,32,-2,0.268717,0.224055,0.142022,0.044662,0.126695,0.481769,0.531599,0.573460,0.421065,NaN,NaN,NaN,NaN
4,Llama1B train seed42 k32 layer-1,Llama1B,train,42,32,32,-1,0.282780,0.239883,0.140226,0.042897,0.142554,0.479902,0.519763,2.558987,3.516439,1.0,0.96875,0.96875,0.96875


In [8]:
headline = {
    'num_runs': len(table_df),
    'wins_selected_neg_over_no_neg': int((table_df['delta_neg_minus_no_neg'] > 0).sum()),
    'mean_delta': float(table_df['delta_neg_minus_no_neg'].mean()),
    'median_delta': float(table_df['delta_neg_minus_no_neg'].median()),
    'same_answer_better_count': int((table_df['selected_neg_same_answer'] < table_df['selected_no_neg_same_answer']).sum()),
    'mean_jaccard': float(overlap_df['jaccard'].mean()) if len(overlap_df) else float('nan'),
}
headline

{'num_runs': 5,
 'wins_selected_neg_over_no_neg': 5,
 'mean_delta': 0.050949716567993165,
 'median_delta': 0.044661909341812134,
 'same_answer_better_count': 5,
 'mean_jaccard': 0.14415113801548884}

## 2. 原始总表

这里显示的是 notebook 刚刚从原始数据和模型重新计算出来的结果，不依赖已有 CSV。


In [9]:
cols = [
    'label', 'model', 'split', 'seed', 'top_k', 'num_samples', 'layer_index',
    'selected_neg_omsa', 'selected_no_neg_omsa', 'delta_neg_minus_no_neg', 'random_omsa',
    'selected_neg_same_answer', 'selected_no_neg_same_answer',
    'selected_neg_logit_drop', 'selected_no_neg_logit_drop',
    'main_accuracy', 'socratic_accuracy', 'rewrite_consistency',
]
table_df[cols]

,label,model,split,seed,top_k,num_samples,layer_index,selected_neg_omsa,selected_no_neg_omsa,delta_neg_minus_no_neg,random_omsa,selected_neg_same_answer,selected_no_neg_same_answer,selected_neg_logit_drop,selected_no_neg_logit_drop,main_accuracy,socratic_accuracy,rewrite_consistency
0,Llama1B test seed42 k32 layer-1,Llama1B,test,42,32,16,-1,0.284572,0.228589,0.055983,0.111626,0.479293,0.541445,3.117866,3.287113,NaN,NaN,NaN
1,Llama1B train seed7 k32 layer-1,Llama1B,train,7,32,32,-1,0.264731,0.246013,0.018719,0.101140,0.491194,0.509678,3.373740,3.644687,NaN,NaN,NaN
2,Llama1B train seed42 k16 layer-1,Llama1B,train,42,16,32,-1,0.277310,0.184822,0.092488,0.153060,0.481921,0.579034,1.598073,2.105577,NaN,NaN,NaN
3,Llama1B train seed42 k32 layer-2,Llama1B,train,42,32,32,-2,0.268717,0.224055,0.044662,0.142022,0.481769,0.531599,0.573460,0.421065,NaN,NaN,NaN
4,Llama1B train seed42 k32 layer-1,Llama1B,train,42,32,32,-1,0.282780,0.239883,0.042897,0.140226,0.479902,0.519763,2.558987,3.516439,1.0,0.96875,0.96875


## 3. 按模型分开看

最值得检查的是：
- `delta_neg_minus_no_neg` 是否始终为正
- `selected_neg_same_answer` 是否始终更低


In [10]:
table_df.groupby('model')['delta_neg_minus_no_neg'].agg(['count', 'mean', 'median', 'min', 'max']).reset_index()

,model,count,mean,median,min,max
0,Llama1B,5,0.05095,0.044662,0.018719,0.092488


In [11]:
for model_name in sorted(table_df['model'].unique()):
    print('\n===', model_name, '===')
    display(
        table_df.loc[table_df['model'] == model_name, [
            'label',
            'selected_neg_omsa',
            'selected_no_neg_omsa',
            'delta_neg_minus_no_neg',
            'selected_neg_same_answer',
            'selected_no_neg_same_answer',
        ]].sort_values('label').reset_index(drop=True)
    )


=== Llama1B ===


,label,selected_neg_omsa,selected_no_neg_omsa,delta_neg_minus_no_neg,selected_neg_same_answer,selected_no_neg_same_answer
0,Llama1B test seed42 k32 layer-1,0.284572,0.228589,0.055983,0.479293,0.541445
1,Llama1B train seed42 k16 layer-1,0.277310,0.184822,0.092488,0.481921,0.579034
2,Llama1B train seed42 k32 layer-1,0.282780,0.239883,0.042897,0.479902,0.519763
3,Llama1B train seed42 k32 layer-2,0.268717,0.224055,0.044662,0.481769,0.531599
4,Llama1B train seed7 k32 layer-1,0.264731,0.246013,0.018719,0.491194,0.509678


In [12]:
checks = pd.DataFrame({
    'all_delta_positive': [(table_df['delta_neg_minus_no_neg'] > 0).all()],
    'all_same_answer_improved': [(table_df['selected_neg_same_answer'] < table_df['selected_no_neg_same_answer']).all()],
    'mean_delta_llama1b': [table_df.loc[table_df['model'] == 'Llama1B', 'delta_neg_minus_no_neg'].mean() if 'Llama1B' in table_df['model'].values else np.nan],
    'mean_delta_gpt2': [table_df.loc[table_df['model'] == 'gpt2', 'delta_neg_minus_no_neg'].mean() if 'gpt2' in table_df['model'].values else np.nan],
})
checks

,all_delta_positive,all_same_answer_improved,mean_delta_llama1b,mean_delta_gpt2
0,True,True,0.05095,NaN


## 4. 坐标重叠分析

这一节用同样重跑出来的 `selected_neg` 坐标集合计算 overlap。
如果 Jaccard 不高，并不一定是坏事，通常表示稳定的是统计判别准则，不是固定坐标 id。


In [13]:
overlap_df.groupby('model')['jaccard'].agg(['count', 'mean', 'median', 'min', 'max']).reset_index() if len(overlap_df) else overlap_df

,model,count,mean,median,min,max
0,Llama1B,10,0.144151,0.116279,0.032258,0.5


In [14]:
overlap_df.sort_values(['model', 'jaccard'], ascending=[True, False]).reset_index(drop=True).head(20) if len(overlap_df) else overlap_df

,run_a,run_b,model,intersection,union,jaccard
0,Llama1B train seed42 k32 layer-1,Llama1B train seed42 k16 layer-1,Llama1B,16,32,0.500000
1,Llama1B train seed42 k32 layer-1,Llama1B train seed7 k32 layer-1,Llama1B,9,55,0.163636
2,Llama1B train seed7 k32 layer-1,Llama1B test seed42 k32 layer-1,Llama1B,9,55,0.163636
3,Llama1B train seed42 k32 layer-1,Llama1B test seed42 k32 layer-1,Llama1B,7,57,0.122807
4,Llama1B train seed42 k16 layer-1,Llama1B train seed42 k32 layer-2,Llama1B,5,43,0.116279
5,Llama1B train seed7 k32 layer-1,Llama1B train seed42 k16 layer-1,Llama1B,5,43,0.116279
6,Llama1B train seed42 k32 layer-1,Llama1B train seed42 k32 layer-2,Llama1B,6,58,0.103448
7,Llama1B train seed42 k16 layer-1,Llama1B test seed42 k32 layer-1,Llama1B,4,44,0.090909
8,Llama1B train seed42 k32 layer-2,Llama1B test seed42 k32 layer-1,Llama1B,2,62,0.032258
9,Llama1B train seed7 k32 layer-1,Llama1B train seed42 k32 layer-2,Llama1B,2,62,0.032258


## 5. 查看原始 parquet 样本

下面这格不依赖任何现成 summary，而是直接从原始 parquet 里查看同一道题在 `main` 和 `socratic` 下的文本。


In [15]:
main_df = pd.read_parquet(DATASET_ROOT / 'gsm8k' / 'main' / 'train-00000-of-00001.parquet')
soc_df = pd.read_parquet(DATASET_ROOT / 'gsm8k' / 'socratic' / 'train-00000-of-00001.parquet')

idx = 0
print('QUESTION:')
print(main_df.iloc[idx]['question'])
print('\nMAIN ANSWER RAW:')
print(main_df.iloc[idx]['answer'])
print('\nSOCRATIC ANSWER RAW:')
print(soc_df.iloc[idx]['answer'])

QUESTION:
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

MAIN ANSWER RAW:
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72

SOCRATIC ANSWER RAW:
How many clips did Natalia sell in May? ** Natalia sold 48/2 = <<48/2=24>>24 clips in May.
How many clips did Natalia sell altogether in April and May? ** Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72


## 6. 查看重跑时的真实样本

这里直接从 `run_outputs` 里拿真实样本，而不是从现成 `summary.json` 读取。
如果某个 config 开了 `do_generation_eval=True`，这里还会带上生成结果。


In [16]:
sample_rows = []
for out in run_outputs:
    for ex in out['features'][:3]:
        sample_rows.append({
            'source_run': out['row']['label'],
            'question': ex.question,
            'answer': ex.answer,
            'pred_main': ex.pred_main,
            'pred_socratic': ex.pred_socratic,
            'generation_main': ex.generation_main,
            'generation_socratic': ex.generation_socratic,
        })

sample_df = pd.DataFrame(sample_rows)
sample_df.head(10)

,source_run,question,answer,pred_main,pred_socratic,generation_main,generation_socratic
0,Llama1B train seed42 k32 layer-1,Stefan goes to a restaurant to eat dinner with...,108,108,108,The final answer is $108. I hope it is correct.,The final answer is $108. I hope it is correct.
1,Llama1B train seed42 k32 layer-1,The gauge on a water tank shows that the tank ...,24,24,24,24,The tank holds 24 gallons of water when full.
2,Llama1B train seed42 k32 layer-1,Ben has 8 apples more than Phillip does. Tom h...,18,18,18,The final answer is 18.,The final answer is 18.
3,Llama1B train seed7 k32 layer-1,Two white socks cost 25 cents more than a sing...,3,None,None,None,None
4,Llama1B train seed7 k32 layer-1,There are 55 people at the track meet. 30 of t...,10,None,None,None,None
5,Llama1B train seed7 k32 layer-1,George and Harry want to fill a pool with buck...,22,None,None,None,None
6,Llama1B train seed42 k16 layer-1,Stefan goes to a restaurant to eat dinner with...,108,None,None,None,None
7,Llama1B train seed42 k16 layer-1,The gauge on a water tank shows that the tank ...,24,None,None,None,None
8,Llama1B train seed42 k16 layer-1,Ben has 8 apples more than Phillip does. Tom h...,18,None,None,None,None
9,Llama1B train seed42 k32 layer-2,Stefan goes to a restaurant to eat dinner with...,108,None,None,None,None


## 7. 可选导出

如果你希望把 notebook 当前重跑出的结果另存成文件，再执行这一格。
这一步是可选的，主分析不依赖它。


In [17]:
# out_table = ROOT / 'pilot_results' / 'notebook_rerun_non_training_validation.csv'
# out_overlap = ROOT / 'pilot_results' / 'notebook_rerun_non_training_overlap.csv'
# table_df.to_csv(out_table, index=False)
# overlap_df.to_csv(out_overlap, index=False)
# print(out_table)
# print(out_overlap)

## 8. 读 notebook 时建议重点看什么

建议重点检查这三句是否成立：

1. `selected-neg` 是否稳定优于 `selected-no-neg`。
2. 这种优势是否主要来自更低的 `SameAnswerStability`，也就是更少 answer cue 污染。
3. Strong model (`Llama1B`) 上，这个趋势是否比 `gpt2` 更清楚。

如果这三句都成立，那么 analysis-only motivation 就是站得住的。
